# Day 7 · Lab 1: Model Context Protocol (MCP) Foundations & Primitives
### Production AI Frameworks Curriculum · Enterprise AI Engineering · Udaan Learning

---

## 🎯 Learning Objectives
1. Understand why MCP solves the **M × N Integration Problem** (converting disparate APIs into a universal standard).
2. Deconstruct the roles of **Hosts**, **Clients**, and **Servers**.
3. Master the 3 MCP Primitives: **Tools** (callable actions), **Resources** (read-only context payloads), and **Prompts** (slash command templates).
4. Analyze the **JSON-RPC 2.0** wire protocol under the hood.
5. Explore the **2026 MCP Specification** (Tasks, Generative UI MCP Apps, and OAuth 2.1).



In [ ]:
import json
from typing import Dict, Any, List

print("[*] Lab 1: MCP Foundations loaded.")


---
## 1. The M × N Integration Nightmare vs. The MCP Standard
Before MCP, connecting M models to N data sources required M × N custom plugins.
With MCP, every data source builds **one MCP Server**, and every AI application acts as an **MCP Client** (M + N simplicity).


In [ ]:
def calculate_integration_cost(num_models: int, num_data_sources: int) -> Dict[str, int]:
    bespoke_plugins = num_models * num_data_sources
    mcp_connectors = num_models + num_data_sources
    savings_pct = int(((bespoke_plugins - mcp_connectors) / bespoke_plugins) * 100)
    return {
        "bespoke_plugins_required": bespoke_plugins,
        "mcp_standard_connectors": mcp_connectors,
        "complexity_reduction_pct": savings_pct
    }

# Enterprise scenario: 5 AI tools / models across 20 enterprise data sources
res = calculate_integration_cost(5, 20)
print(f"Without MCP: {res['bespoke_plugins_required']} custom plugins.")
print(f"With MCP:    {res['mcp_standard_connectors']} standard connectors.")
print(f"Complexity Reduction: {res['complexity_reduction_pct']}% fewer interfaces to maintain!")


---
## 2. The Wire Protocol: JSON-RPC 2.0 In Action
All MCP communications conform to the **JSON-RPC 2.0** specification.
Let's inspect the fundamental message exchange:
1. `tools/list`: Client discovers server capabilities.
2. `tools/call`: Client executes a tool with structured arguments.
3. Server returns text, image, or structured JSON content.


In [ ]:
json_rpc_tools_list_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "tools": [
            {
                "name": "query_patient_prescriptions",
                "description": "Fetch authorized EHR prescription records for an admitted patient.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "patient_name": {"type": "string", "description": "Full name of the patient"},
                        "mrn": {"type": "string", "description": "Medical Record Number (e.g., MRN882910)"}
                    },
                    "required": ["patient_name"]
                }
            },
            {
                "name": "lookup_drug_interactions",
                "description": "Check contraindications and drug-drug interactions in the Knowledge Graph.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "medication_name": {"type": "string", "description": "Generic drug name"}
                    },
                    "required": ["medication_name"]
                }
            }
        ]
    }
}

print("=== DISCOVERED MCP TOOLS ===")
for tool in json_rpc_tools_list_response["result"]["tools"]:
    print(f"• Tool: {tool['name']}")
    print(f"  Description: {tool['description']}")
    print(f"  Required Inputs: {tool['inputSchema']['required']}")


---
## 3. The 3 MCP Primitives: Tools, Resources, and Prompts
| Primitive | Purpose | Interaction Mode | Example |
| :--- | :--- | :--- | :--- |
| **Tool** | Execute an action with side effects | LLM calls dynamically; requires confirmation | `query_patient_prescriptions(name='Rahul')` |
| **Resource** | Read-only contextual payload | Client attaches directly as context | `ehr://patients/MRN882910/vitals` |
| **Prompt** | Reusable workflow template | User triggers as a slash shortcut | `/reconcile-patient-chart patient_id=MRN882910` |



In [ ]:
mcp_resource_payload = {
    "uri": "ehr://patients/MRN882910/vitals",
    "mimeType": "application/json",
    "name": "Rahul Sharma - Admitted Vitals",
    "text": json.dumps({
        "patient": "Rahul Sharma",
        "mrn": "MRN882910",
        "ward": "Endocrinology",
        "bp": "128/82",
        "glucose_fasting": "142 mg/dL",
        "timestamp": "2026-10-30T07:15:00Z"
    }, indent=2)
}

print("=== MCP RESOURCE PAYLOAD ===")
print("Resource URI:", mcp_resource_payload["uri"])
print("Content:
", mcp_resource_payload["text"])


---
## 💡 Summary & Key Takeaways
1. MCP acts as the **universal standard** connecting AI models to tools and context.
2. Hosts run clients; clients connect to servers over **stdio** (local) or **SSE** (remote).
3. The protocol is strictly **JSON-RPC 2.0**.
4. In **Lab 2**, we will write a complete **Custom MCP Server in Python**!

